In [2]:

import numpy as np
import pandas as pd

from datetime import datetime, timedelta


import duckdb


DB_PATH = r"C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\duckdb\nhl_data.duckdb"

con = duckdb.connect(DB_PATH)
print("Connected:", DB_PATH)


# Suppress SettingWithCopyWarning
pd.options.mode.chained_assignment = None
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

Connected: C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\duckdb\nhl_data.duckdb


In [3]:

con.close()
print("DuckDB connection closed.")

DuckDB connection closed.


In [24]:

con.execute("SHOW TABLES").fetchdf()

,name
0,all_clinch_history
1,all_final_standings
2,all_game_summary
3,all_game_summary_backup
4,all_goalie_games
5,all_pk
6,all_play_by_play
7,all_play_by_play_1
8,all_pp
9,all_prediction_data_agg


In [25]:
teams = con.execute('select * from all_teams').df()
teams.head()

,team_id,tri_code
0,24,ANA
1,53,ARI
2,6,BOS
3,7,BUF
4,12,CAR


## Game Summary

In [26]:
history_game_summary=con.execute("""SELECT *  
                                 FROM all_game_summary 
                                 WHERE season_id != 20252026
                                 ORDER BY game_id""").df()

history_game_summary.tail()

,game_id,season_id,away_team,away_tri_code,home_team,home_tri_code,awayTeam.score,homeTeam.score,gameState,link,game_date,api_snapshot_date
20286,2024021308,20242025,17,DET,10,TOR,3,4,OFF,https://api-web.nhle.com/v1/gamecenter/2024021308/play-by-play,2025-04-17,2025-12-09
20287,2024021309,20242025,12,CAR,9,OTT,5,7,OFF,https://api-web.nhle.com/v1/gamecenter/2024021309/play-by-play,2025-04-17,2025-12-09
20288,2024021310,20242025,14,TBL,3,NYR,0,4,OFF,https://api-web.nhle.com/v1/gamecenter/2024021310/play-by-play,2025-04-17,2025-12-09
20289,2024021311,20242025,15,WSH,5,PIT,2,5,OFF,https://api-web.nhle.com/v1/gamecenter/2024021311/play-by-play,2025-04-17,2025-12-09
20290,2024021312,20242025,2,NYI,29,CBJ,1,6,OFF,https://api-web.nhle.com/v1/gamecenter/2024021312/play-by-play,2025-04-17,2025-12-09


In [27]:
game_summary=con.execute("""SELECT *  
                         FROM all_game_summary
                         WHERE season_id = 20252026 
                         ORDER BY game_id""").df()

game_summary.tail()

,game_id,season_id,away_team,away_tri_code,home_team,home_tri_code,awayTeam.score,homeTeam.score,gameState,link,game_date,api_snapshot_date
1307,2025021308,20252026,28,SJS,52,WPG,<NA>,<NA>,FUT,https://api-web.nhle.com/v1/gamecenter/2025021308/play-by-play,2026-04-16,2025-12-09
1308,2025021309,20252026,19,STL,68,UTA,<NA>,<NA>,FUT,https://api-web.nhle.com/v1/gamecenter/2025021309/play-by-play,2026-04-16,2025-12-09
1309,2025021310,20252026,26,LAK,20,CGY,<NA>,<NA>,FUT,https://api-web.nhle.com/v1/gamecenter/2025021310/play-by-play,2026-04-16,2025-12-09
1310,2025021311,20252026,23,VAN,22,EDM,<NA>,<NA>,FUT,https://api-web.nhle.com/v1/gamecenter/2025021311/play-by-play,2026-04-16,2025-12-09
1311,2025021312,20252026,55,SEA,21,COL,<NA>,<NA>,FUT,https://api-web.nhle.com/v1/gamecenter/2025021312/play-by-play,2026-04-16,2025-12-09


In [28]:
clinch_date = con.execute("""
                          SELECT
                          season_id,
                          clinch_date as game_date,
                          tri_code,
                          clinch_type
                          FROM all_clinch_history 
                          ORDER BY season_id""").df()
clinch_outcomes = clinch_date.assign(
    made_playoffs=lambda df: df["clinch_type"].eq("x").astype(int)
)




## Standings

In [29]:
history_daily_standings=con.execute("""
                                    SELECT *  
                                    FROM all_standings_daily
                                    WHERE season_id !=20252026""").df()


history_daily_standings=history_daily_standings.merge(clinch_date, on=['season_id', 'game_date', 'tri_code'], how='left')

history_daily_standings=history_daily_standings.merge(
    clinch_outcomes[["season_id", "tri_code", "made_playoffs"]],
    on=["season_id", "tri_code"],
    how="left"
)
history_daily_standings=history_daily_standings[[
    'game_date', 'season_id', 'team_id','tri_code', 'team_name',
    
    'games_remaining', 
    'wins', 'wins_pctg', 'losses', 'ot_losses',
    'rotw', 'rotw_pctg', 'rw',  'so_losses', 'so_wins',
    
    'points', 'points_pct','projected_points', 'in_playoff_spot', 
    'goals_for', 'gf_pctg','goals_against',
    'goal_diff', 'goal_diff_pctg',
    'streak', 'streak_count', 
    
    'last_10_gp','last_10_wins', 'last_10_losses', 'last_10_otl', 
    'last_10_points', 
    'last_10_gf', 'last_10_ga', 'last_10_goal_diff', 'last_10_rotw',
    'last_10_rw', 
     
       
    'home_gp', 
    'home_wins', 'home_losses', 'home_otlosses',
    'home_rotw', 'home_rw',
    'home_points', 'home_gf','home_ga',  'home_goal_diff',
    
    'road_gp', 
    'road_wins', 'road_losses', 'road_ot_losses', 
    'road_rotw','road_rw',
    'road_points','road_gf', 'road_ga',  'road_goal_diff', 
    
    'division', 'division_rank',
    'division_home_rank', 'division_road_rank', 'division_last_10_rank',
    
    'conference', 'conference_rank', 'conference_home_rank',
    'conference_road_rank', 'conference_last_10_rank', 
    
    'league_rank',
    'league_home_rank', 'league_road_rank', 'league_last_10_rank',
    
    'wildcard_rank', "made_playoffs"
    
]]
history_daily_standings['made_playoffs'] = history_daily_standings['made_playoffs'].astype("Int64")
history_daily_standings.head(2)



,game_date,season_id,team_id,tri_code,team_name,games_remaining,wins,wins_pctg,losses,ot_losses,rotw,rotw_pctg,rw,so_losses,so_wins,points,points_pct,projected_points,in_playoff_spot,goals_for,gf_pctg,goals_against,goal_diff,goal_diff_pctg,streak,streak_count,last_10_gp,last_10_wins,last_10_losses,last_10_otl,last_10_points,last_10_gf,last_10_ga,last_10_goal_diff,last_10_rotw,last_10_rw,home_gp,home_wins,home_losses,home_otlosses,home_rotw,home_rw,home_points,home_gf,home_ga,home_goal_diff,road_gp,road_wins,road_losses,road_ot_losses,road_rotw,road_rw,road_points,road_gf,road_ga,road_goal_diff,division,division_rank,division_home_rank,division_road_rank,division_last_10_rank,conference,conference_rank,conference_home_rank,conference_road_rank,conference_last_10_rank,league_rank,league_home_rank,league_road_rank,league_last_10_rank,wildcard_rank,made_playoffs
0,2013-10-01,20132014,16,CHI,Chicago Blackhawks,81,1,1.0,0,0,1,1.0,1,0,0,2,1.0,164,0,6,6.0,4,2,2.0,W,1.0,1,1,0,0,2,6,4,2,1,1,1,1,0,0,1,1,2,6,4,2,0,0,0,0,0,0,0,0,0,0,Central,1,1,2,1,Western,1,1,4,1,1,1,8,1,0,1
1,2013-10-01,20132014,10,TOR,Toronto Maple Leafs,81,1,1.0,0,0,1,1.0,1,0,0,2,1.0,164,0,4,4.0,3,1,1.0,W,1.0,1,1,0,0,2,4,3,1,1,1,0,0,0,0,0,0,0,0,0,0,1,1,0,0,1,1,2,4,3,1,Atlantic,1,7,1,1,Eastern,1,14,1,1,2,25,1,2,0,0


In [30]:
daily_standings=con.execute("""SELECT *  
                            FROM all_standings_daily
                            WHERE season_id =20252026""").df()

daily_standings.head()


,game_date,season_id,tri_code,team_name,games_played,road_gp,division,division_rank,division_home_rank,division_road_rank,division_last_10_rank,conference,conference_rank,conference_home_rank,conference_road_rank,conference_last_10_rank,league_rank,league_home_rank,league_road_rank,league_last_10_rank,wildcard_rank,wins,wins_pctg,rw,losses,ot_losses,rotw,rotw_pctg,road_losses,road_ot_losses,road_rotw,road_rw,points,points_pct,home_points,road_points,goals_for,gf_pctg,home_gf,road_gf,goals_against,home_ga,road_ga,goal_diff,goal_diff_pctg,home_goal_diff,road_goal_diff,streak,streak_count,last_10_gp,last_10_wins,last_10_goal_diff,last_10_ga,last_10_gf,last_10_losses,last_10_otl,last_10_points,last_10_rotw,last_10_rw,home_gp,home_wins,home_losses,home_otlosses,home_rotw,home_rw,road_wins,so_losses,so_wins,in_playoff_spot,games_remaining,ppg,projected_points,team_id
0,2025-10-07,20252026,COL,Colorado Avalanche,1,1,Central,1,2,1,1,Western,1,4,1,1,1,8,1,1,0,1,1.0,1,0,0,1,1.0,0,0,1,1,2,1.0,0,2,4,4.0,0,4,1,0,1,3,3.0,0,3,W,1.0,1,1,3,1,4,0,0,2,1,1,0,0,0,0,0,0,1,0,0,0,81,2.0,164,21
1,2025-10-07,20252026,PIT,Pittsburgh Penguins,1,1,Metropolitan,1,6,1,1,Eastern,1,12,1,1,2,20,2,2,0,1,1.0,1,0,0,1,1.0,0,0,1,1,2,1.0,0,2,3,3.0,0,3,0,0,0,3,3.0,0,3,W,1.0,1,1,3,0,3,0,0,2,1,1,0,0,0,0,0,0,1,0,0,0,81,2.0,164,5
2,2025-10-07,20252026,FLA,Florida Panthers,1,0,Atlantic,1,1,4,1,Eastern,2,1,7,2,3,1,12,3,0,1,1.0,1,0,0,1,1.0,0,0,0,0,2,1.0,2,0,3,3.0,3,0,2,2,0,1,1.0,1,0,W,1.0,1,1,1,2,3,0,0,2,1,1,1,1,0,0,1,1,0,0,0,0,81,2.0,164,13
3,2025-10-07,20252026,ANA,Anaheim Ducks,0,0,Pacific,1,1,1,1,Western,2,1,2,2,4,2,3,4,0,0,0.0,0,0,0,0,0.0,0,0,0,0,0,0.0,0,0,0,0.0,0,0,0,0,0,0,0.0,0,0,None,NaN,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,82,NaN,0,24
4,2025-10-07,20252026,BOS,Boston Bruins,0,0,Atlantic,2,2,1,2,Eastern,3,2,2,3,5,3,4,5,0,0,0.0,0,0,0,0,0.0,0,0,0,0,0,0.0,0,0,0,0.0,0,0,0,0,0,0,0.0,0,0,None,NaN,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,82,NaN,0,6


In [31]:
daily_standings=daily_standings[[
    'game_date', 'season_id', 'team_id','tri_code', 'team_name',
    
    'games_remaining', 
    'wins', 'wins_pctg', 'losses', 'ot_losses',
    'rotw', 'rotw_pctg', 'rw',  'so_losses', 'so_wins',
    
    'points', 'points_pct','projected_points', 'in_playoff_spot', 
    'goals_for', 'gf_pctg','goals_against',
    'goal_diff', 'goal_diff_pctg',
    'streak', 'streak_count', 
    
    'last_10_gp','last_10_wins', 'last_10_losses', 'last_10_otl', 
    'last_10_points', 
    'last_10_gf', 'last_10_ga', 'last_10_goal_diff', 'last_10_rotw',
    'last_10_rw', 
     
       
    'home_gp', 
    'home_wins', 'home_losses', 'home_otlosses',
    'home_rotw', 'home_rw',
    'home_points', 'home_gf','home_ga',  'home_goal_diff',
    
    'road_gp', 
    'road_wins', 'road_losses', 'road_ot_losses', 
    'road_rotw','road_rw',
    'road_points','road_gf', 'road_ga',  'road_goal_diff', 
    
    'division', 'division_rank',
    'division_home_rank', 'division_road_rank', 'division_last_10_rank',
    
    'conference', 'conference_rank', 'conference_home_rank',
    'conference_road_rank', 'conference_last_10_rank', 
    
    'league_rank',
    'league_home_rank', 'league_road_rank', 'league_last_10_rank',
    
    'wildcard_rank',     
    
]]

daily_standings.head(2)

,game_date,season_id,team_id,tri_code,team_name,games_remaining,wins,wins_pctg,losses,ot_losses,rotw,rotw_pctg,rw,so_losses,so_wins,points,points_pct,projected_points,in_playoff_spot,goals_for,gf_pctg,goals_against,goal_diff,goal_diff_pctg,streak,streak_count,last_10_gp,last_10_wins,last_10_losses,last_10_otl,last_10_points,last_10_gf,last_10_ga,last_10_goal_diff,last_10_rotw,last_10_rw,home_gp,home_wins,home_losses,home_otlosses,home_rotw,home_rw,home_points,home_gf,home_ga,home_goal_diff,road_gp,road_wins,road_losses,road_ot_losses,road_rotw,road_rw,road_points,road_gf,road_ga,road_goal_diff,division,division_rank,division_home_rank,division_road_rank,division_last_10_rank,conference,conference_rank,conference_home_rank,conference_road_rank,conference_last_10_rank,league_rank,league_home_rank,league_road_rank,league_last_10_rank,wildcard_rank
0,2025-10-07,20252026,21,COL,Colorado Avalanche,81,1,1.0,0,0,1,1.0,1,0,0,2,1.0,164,0,4,4.0,1,3,3.0,W,1.0,1,1,0,0,2,4,1,3,1,1,0,0,0,0,0,0,0,0,0,0,1,1,0,0,1,1,2,4,1,3,Central,1,2,1,1,Western,1,4,1,1,1,8,1,1,0
1,2025-10-07,20252026,5,PIT,Pittsburgh Penguins,81,1,1.0,0,0,1,1.0,1,0,0,2,1.0,164,0,3,3.0,0,3,3.0,W,1.0,1,1,0,0,2,3,0,3,1,1,0,0,0,0,0,0,0,0,0,0,1,1,0,0,1,1,2,3,0,3,Metropolitan,1,6,1,1,Eastern,1,12,1,1,2,20,2,2,0


## Play by Play

In [32]:
history_pbp=con.execute(""" SELECT
                    game_id,
                    team_id,
                    home_team,
                    away_team,
                    type_code,
                    typeDescKey,
                    zone_code,
                    x_coord,
                    y_coord,
                    period_number,
                    game_in_seconds,
                    goalie,
                    sortOrder,
                    season_id
                    from all_play_by_play 
                    where season_id !=20252026
                    order by game_id, sortOrder
                    """).df()
history_pbp.head()

,game_id,team_id,home_team,away_team,type_code,typeDescKey,zone_code,x_coord,y_coord,period_number,game_in_seconds,goalie,sortOrder,season_id
0,2008020001,3,14,3,506,shot-on-goal,None,NaN,NaN,1,94,8469608,1,20082009
1,2008020001,14,14,3,506,shot-on-goal,None,NaN,NaN,1,155,8468685,2,20082009
2,2008020001,14,14,3,506,shot-on-goal,None,NaN,NaN,1,157,8468685,3,20082009
3,2008020001,3,14,3,506,shot-on-goal,None,NaN,NaN,1,215,8469608,4,20082009
4,2008020001,3,14,3,506,shot-on-goal,None,NaN,NaN,1,225,8469608,5,20082009


In [33]:
current_pbp=con.execute(""" SELECT
                    game_id,
                    team_id,
                    home_team,
                    away_team,
                    type_code,
                    typeDescKey,
                    zone_code,
                    x_coord,
                    y_coord,
                    period_number,
                    game_in_seconds,
                    goalie,
                    sortOrder,
                    season_id
                    from all_play_by_play 
                    where season_id ==20252026
                    order by game_id, sortOrder
                    """).df()
current_pbp.head()

,game_id,team_id,home_team,away_team,type_code,typeDescKey,zone_code,x_coord,y_coord,period_number,game_in_seconds,goalie,sortOrder,season_id
0,2025020001,<NA>,13,16,520,period-start,None,NaN,NaN,1,0,<NA>,8,20252026
1,2025020001,16,13,16,502,faceoff,N,0.0,0.0,1,0,<NA>,11,20252026
2,2025020001,13,13,16,508,blocked-shot,D,-61.0,3.0,1,22,<NA>,12,20252026
3,2025020001,<NA>,13,16,516,stoppage,None,NaN,NaN,1,34,<NA>,13,20252026
4,2025020001,16,13,16,502,faceoff,D,-69.0,22.0,1,34,<NA>,16,20252026


### Rush

In [34]:
def build_rush_stats(df):
    """
    Unified rush-shot classifier and rush-shot stats generator.
    Works for both historical and current play-by-play datasets.
    """

    df = df.copy()

    # --------------------------------------
    # 1. Define event groups
    # --------------------------------------
    shot_events = {
        "shot-on-goal",
        "missed-shot",
        "goal",
        "blocked-shot",
        "failed-shot-attempt",
    }

    transition_events = {"giveaway", "takeaway", "hit"}

    stoppage_events = {
        "faceoff",
        "stoppage",
        "period-end",
        "game-end",
        "penalty",
        "delayed-penalty",
    }

    valid_prev_events = shot_events | transition_events | stoppage_events

    # --------------------------------------
    # 2. Filter to relevant events
    # --------------------------------------
    plays = df[df["typeDescKey"].isin(valid_prev_events)].copy()
    plays = plays.sort_values(["game_id", "sortOrder"])

    # --------------------------------------
    # 3. Build previous-event context
    # --------------------------------------
    g = plays.groupby("game_id", sort=False)

    plays["prev_event_time"] = g["game_in_seconds"].shift(1)
    plays["prev_event_type"] = g["typeDescKey"].shift(1)
    plays["prev_zone"] = g["zone_code"].shift(1)
    plays["prev_team_id"] = g["team_id"].shift(1)

    plays["prev_event_was_stoppage"] = plays["prev_event_type"].isin(stoppage_events)
    plays["time_diff"] = plays["game_in_seconds"] - plays["prev_event_time"]

    # --------------------------------------
    # 4. Rush classification
    # --------------------------------------
    is_shot = plays["typeDescKey"].isin(shot_events)
    quick = plays["time_diff"].between(0, 5, inclusive="both")

    possession_change = (
        plays["prev_team_id"].notna()
        & (plays["prev_team_id"] != plays["team_id"])
    )

    same_team_from_dz_nz = (
        (plays["prev_team_id"] == plays["team_id"])
        & plays["prev_zone"].isin(["D", "N"])
    )

    no_stoppage = ~plays["prev_event_was_stoppage"]

    rush_mask = (
        is_shot
        & quick
        & no_stoppage
        & (possession_change | same_team_from_dz_nz)
    )

    rush_events = plays.loc[rush_mask].copy().drop_duplicates()


    
    rush_events["rush_shot"] = 1

    # --------------------------------------
    # 5. Rush shots FOR
    # --------------------------------------
    rush_for = (
        rush_events
        .groupby(["season_id", "game_id", "team_id"])
        .size()
        .reset_index(name="rush_shots_for")
    )

    # --------------------------------------
    # 6. Rush shots AGAINST
    # --------------------------------------
    rush_against = (
        rush_events
        .groupby(["season_id", "game_id", "prev_team_id"])
        .size()
        .reset_index(name="rush_shots_against")
        .rename(columns={"prev_team_id": "team_id"})
    )

    # --------------------------------------
    # 7. Merge FOR + AGAINST
    # --------------------------------------
    rush_stats = (
        rush_for
        .merge(rush_against, on=["season_id", "game_id", "team_id"], how="inner")
        .fillna(0)
    )

    rush_stats["rush_shots_for"] = rush_stats["rush_shots_for"].astype(int)
    rush_stats["rush_shots_against"] = rush_stats["rush_shots_against"].astype(int)

    rush_stats["rush_shot_differential"] = (
        rush_stats["rush_shots_for"] - rush_stats["rush_shots_against"]
    )

    rush_stats = (
        rush_stats
        .sort_values(["game_id", "rush_shot_differential"], ascending=[True, False])
        .reset_index(drop=True)
    )
    
    return rush_events, rush_stats


In [35]:

# Usage
# hist_rush_events, hist_rush_stats = build_rush_stats(history_pbp)
curr_rush_events, curr_rush_stats = build_rush_stats(current_pbp)


In [36]:



curr_rush_stats.tail()

,season_id,game_id,team_id,rush_shots_for,rush_shots_against,rush_shot_differential
1804,20252026,2025020907,9,3,6,-3
1805,20252026,2025020908,18,7,3,4
1806,20252026,2025020908,15,3,7,-4
1807,20252026,2025020909,26,9,6,3
1808,20252026,2025020909,54,6,9,-3


In [ ]:
curr_rush_stats.head()

##### Cumulative Rush Stats

In [ ]:
def cumulative_rush(df):
    rush_to_date = df.copy()
    rush_to_date = rush_to_date.drop_duplicates()
    # Ensure correct ordering
    rush_to_date = rush_to_date.sort_values(
        ["season_id", "team_id", "game_date", "game_id"]
        if "game_date" in rush_to_date.columns
        else ["season_id", "team_id", "game_id"]
    )

    # Group by season + team (NOT game)
    g = rush_to_date.groupby(["season_id", "team_id"])

    # Cumulative totals
    rush_to_date["c_rush_shots_for"] = g["rush_shots_for"].cumsum()
    rush_to_date["c_rush_shots_against"] = g["rush_shots_against"].cumsum()
    rush_to_date["c_rush_diff"] = g["rush_shot_differential"].cumsum()



    # Per-game rates
    rush_to_date["rush_for_per_game"] = (
        rush_to_date["c_rush_shots_for"] / rush_to_date["games_played"]
    )

    rush_to_date["rush_against_per_game"] = (
        rush_to_date["c_rush_shots_against"] / rush_to_date["games_played"]
    )

    rush_to_date["rush_diff_per_game"] = (
        rush_to_date["c_rush_diff"] / rush_to_date["games_played"]
    )

    return rush_to_date



In [ ]:
current_cumulative_rush = cumulative_rush(curr_rush_stats)
# history_cumulative_rush = cumulative_rush(hist_rush_stats)

current_cumulative_rush.head()

### All Shot Attempts

In [43]:
def extract_shot_attempts(pbp_df):
    SHOT_CODES = [505, 506, 507, 508, 537]

    # -------------------------------------------------
    # 1. Filter to shot attempts (PBP-level)
    # -------------------------------------------------
    shot_events = pbp_df.copy()
    shot_events = shot_events[shot_events["type_code"].isin(SHOT_CODES)].copy()
    shot_events = shot_events.sort_values(["game_id", "sortOrder"])

    shot_events["shot_attempt"] = 1

    # Ensure ordering for prexv_team logic

    # -------------------------------------------------
    # 2. Previous team (for against calculation)
    # -------------------------------------------------
    # g = shot_events.groupby("game_id")
    # shot_events["prev_team_id"] = g["team_id"].shift(1)
    teams = (
        shot_events[["season_id", "game_id", "team_id"]]
        .drop_duplicates()
    )

    opponents = (
        teams
        .merge(
            teams,
            on=["season_id", "game_id"],
            suffixes=("", "_opp")
        )
        .query("team_id != team_id_opp")
        .rename(columns={"team_id_opp": "opponent_team_id"})
    )
    
    shot_events = shot_events.merge(
        opponents,
        on=["season_id", "game_id", "team_id"],
        how="left"
    )
    
    bad_games = (
        opponents
        .groupby(["season_id","game_id"])
        .size()
        .reset_index(name="rows")
        .query("rows != 2")
    )

    if not bad_games.empty:
        raise ValueError("Invalid team count in some games")
    
    
    # # -------------------------------------------------
    # # 3. Merge rush flags
    # # -------------------------------------------------
    # rush_small = rush_events_df[
    #     ["game_id", "sortOrder", "game_in_seconds", "rush_shot"]
    # ].copy()

    # shot_events = shot_events.merge(
    #     rush_small,
    #     on=["game_id", "sortOrder", "game_in_seconds"],
    #     how="left"
    # )

    # shot_events["rush_shot"] = shot_events["rush_shot"].fillna(0).astype(int)

    # =================================================
    # GAME-LEVEL AGGREGATION (MINI DF)
    # =================================================

    # -------------------------------------------------
    # 4. Shot attempts FOR
    # -------------------------------------------------
    shots_for = (
        shot_events.groupby(["season_id", "game_id", "team_id"])
                 .size()
                 .reset_index(name="shot_attempts_for")
    )

    # -------------------------------------------------
    # 5. Shot attempts AGAINST
    # -------------------------------------------------
    shots_against = (
        shot_events[shot_events["opponent_team_id"].notna()]
        .groupby(["season_id", "game_id", "opponent_team_id"])
        .size()
        .reset_index(name="shot_attempts_against")
        .rename(columns={"opponent_team_id": "team_id"})
    )

    # -------------------------------------------------
    # 6. Merge + differential
    # -------------------------------------------------
    shot_stats = (
        shots_for
        .merge(
            shots_against,
            on=["season_id", "game_id", "team_id"],
            how="outer"
        )
        .fillna(0)
    )

    shot_stats["shot_attempts_for"] = shot_stats["shot_attempts_for"].astype(int)
    shot_stats["shot_attempts_against"] = shot_stats["shot_attempts_against"].astype(int)

    shot_stats["shot_attempts_differential"] = (
        shot_stats["shot_attempts_for"] - shot_stats["shot_attempts_against"]
    )

    # -------------------------------------------------
    # 7. Corsi For / Against
    # -------------------------------------------------
    corsi_for = (
        shot_events[shot_events["zone_code"] == "O"]
        .groupby(["season_id", "game_id", "team_id"])
        .size()
        .reset_index(name="corsi_for")
    )

    corsi_against = (
        shot_events[
            (shot_events["zone_code"] == "O") &
            (shot_events["opponent_team_id"].notna())
        ]
        .groupby(["season_id", "game_id", "opponent_team_id"])
        .size()
        .reset_index(name="corsi_against")
        .rename(columns={"opponent_team_id": "team_id"})
    )
    

    shot_stats = (
        shot_stats
        .merge(corsi_for, on=["season_id", "game_id", "team_id"], how="left")
        .merge(corsi_against, on=["season_id", "game_id", "team_id"], how="left")
        .fillna(0)
    )

    shot_stats['corsi_diff'] = shot_stats['corsi_for']-shot_stats['corsi_against']
    
    shot_stats = shot_stats.sort_values(
        ["season_id", "team_id", "game_id"]
    )

    shot_stats["games_played"] = (
        shot_stats
        .groupby(["season_id", "team_id"])
        .cumcount() + 1
    )

    # -------------------------------------------------
    # RETURN BOTH
    # -------------------------------------------------
    return shot_stats, shot_events


In [44]:

# hist_shots_stats, hist_shot_events = extract_shot_attempts(history_pbp)
curr_shots_stats, curr_shot_events     = extract_shot_attempts(current_pbp)
curr_shots_stats.head()

,season_id,game_id,team_id,shot_attempts_for,shot_attempts_against,shot_attempts_differential,corsi_for,corsi_against,corsi_diff,games_played
26,20252026,2025020014,1,45,73,-28,30,51,-21,1
50,20252026,2025020026,1,64,54,10,43,28,15,2
88,20252026,2025020045,1,56,70,-14,40,49,-9,3
126,20252026,2025020064,1,67,59,8,48,38,10,4
154,20252026,2025020078,1,51,57,-6,36,40,-4,5


In [ ]:
hist_shots_stats.head()

In [ ]:
curr_shots_stats[curr_shots_stats['game_id']==2025020014]

#### Cumulative Shot Attempts

In [ ]:
def cumulative_shot_attempts(df):
    shots_to_date = df.copy()

    # -------------------------------------------------
    # Ensure required columns exist
    # -------------------------------------------------
    required_cols = [
        "shot_attempts_for", "shot_attempts_against",
        "shot_attempts_differential",
        "corsi_for", "corsi_against"
    ]
    for col in required_cols:
        if col not in shots_to_date.columns:
            shots_to_date[col] = 0

    # -------------------------------------------------
    # Ensure correct ordering
    # -------------------------------------------------
    shots_to_date = shots_to_date.sort_values(
        ["season_id", "team_id", "game_id"]
    )

    # -------------------------------------------------
    # Group by season + team
    # -------------------------------------------------
    g = shots_to_date.groupby(["season_id", "team_id"])

    # -------------------------------------------------
    # Cumulative totals
    # -------------------------------------------------
    shots_to_date["c_shots_for"] = g["shot_attempts_for"].cumsum()
    shots_to_date["c_shots_against"] = g["shot_attempts_against"].cumsum()
    shots_to_date["c_shots_diff"] = g["shot_attempts_differential"].cumsum()

    shots_to_date["c_corsi_for"] = g["corsi_for"].cumsum()
    shots_to_date["c_corsi_against"] = g["corsi_against"].cumsum()
    shots_to_date["c_corsi_diff"] = (
        shots_to_date["c_corsi_for"] - shots_to_date["c_corsi_against"]
    )


    # -------------------------------------------------
    # Per-game rates
    # -------------------------------------------------
    shots_to_date["shot_attempts_for_per_game"] = np.where(
        shots_to_date["games_played"] > 0,
        shots_to_date["c_shots_for"] / shots_to_date["games_played"],
        0.0
    )

    shots_to_date["shot_attempts_against_per_game"] = np.where(
        shots_to_date["games_played"] > 0,
        shots_to_date["c_shots_against"] / shots_to_date["games_played"],
        0.0
    )

    shots_to_date["shots_diff_per_game"] = np.where(
        shots_to_date["games_played"] > 0,
        shots_to_date["c_shots_diff"] / shots_to_date["games_played"],
        0.0
    )

    shots_to_date["corsi_diff_per_game"] = np.where(
        shots_to_date["games_played"] > 0,
        shots_to_date["c_corsi_diff"] / shots_to_date["games_played"],
        0.0
    )

    return shots_to_date


In [ ]:
curr_cumulative_shots = cumulative_shot_attempts(curr_shots_stats)
hist_cumulative_shots = cumulative_shot_attempts(hist_shots_stats)

In [ ]:

curr_cumulative_shots.head()

### Danger Zones

In [45]:

#This is just identifying the location of shot events to determine their danger zone 

def preprocess_shot_dataframe(df):
    required_cols = {
        "x_coord", "y_coord", "game_id",
        "period_number", "game_in_seconds", 'sortOrder'
    }
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    df = df.copy()

    # -----------------------------
    # 1. Boolean danger regions
    # -----------------------------
    hd = (
        ((df["x_coord"] >= 66.5) & (df["x_coord"] <= 89) &
         (df["y_coord"].between(-7, 7)))
        |
        ((df["x_coord"].between(-89, -66.5)) &
         (df["y_coord"].between(-7, 7)))
    )

    md = (
        # Right side
        ((df["x_coord"].between(69, 89)) & df["y_coord"].between(-22, -7))
        |
        ((df["x_coord"].between(54, 69)) & df["y_coord"].between(-22, 22))
        |
        ((df["x_coord"].between(40, 54)) & df["y_coord"].between(-7, 7))
        |
        # Left side mirrored
        ((df["x_coord"].between(-89, -69)) & df["y_coord"].between(-22, -7))
        |
        ((df["x_coord"].between(-69, -54)) & df["y_coord"].between(-22, 22))
        |
        ((df["x_coord"].between(-54, -40)) & df["y_coord"].between(-7, 7))
    )

    # Low danger = not HD or MD
    ld = ~(hd | md)

    # -----------------------------
    # 2. Assign numeric danger level
    # -----------------------------
    df["hd_shot"] = hd.astype(int)
    df["md_shot"] = md.astype(int)
    df["ld_shot"] = ld.astype(int)

    # -----------------------------
    # 3. Timing features
    # -----------------------------
    df = df.sort_values(by=['game_id', 'sortOrder'])
    
    df["game_time"] = (df["period_number"] - 1) * 1200 + df["game_in_seconds"]

    df["time_since_last_shot"] = df.groupby("game_id")["game_time"].diff()

    df["is_rebound"] = (df["time_since_last_shot"] <= 3).astype(int)

    return df


In [46]:
# These are shot attempt events
# hist_danger_event = preprocess_shot_dataframe(hist_shot_events)
curr_danger_event = preprocess_shot_dataframe(curr_shot_events)
curr_danger_event.head()



,game_id,team_id,home_team,away_team,type_code,typeDescKey,zone_code,x_coord,y_coord,period_number,game_in_seconds,goalie,sortOrder,season_id,shot_attempt,opponent_team_id,hd_shot,md_shot,ld_shot,game_time,time_since_last_shot,is_rebound
0,2025020001,13,13,16,508,blocked-shot,D,-61.0,3.0,1,22,<NA>,12,20252026,1,16,0,1,0,22,NaN,0
1,2025020001,13,13,16,508,blocked-shot,D,-64.0,-16.0,1,73,<NA>,25,20252026,1,16,0,1,0,73,51.0,0
2,2025020001,13,13,16,506,shot-on-goal,O,-58.0,-22.0,1,76,8481519,29,20252026,1,16,0,1,0,76,3.0,1
3,2025020001,13,13,16,506,shot-on-goal,O,-33.0,-19.0,1,95,8481519,35,20252026,1,16,0,0,1,95,19.0,0
4,2025020001,16,13,16,507,missed-shot,O,55.0,-29.0,1,104,8475683,38,20252026,1,13,0,0,1,104,9.0,0


In [47]:
def build_enriched_shot_events(
    shot_events: pd.DataFrame,
    danger_events: pd.DataFrame,
    rush_events: pd.DataFrame,
    ) -> pd.DataFrame:
    key_cols = ["season_id", "game_id", "team_id", "sortOrder"]

    combined = (
        shot_events
        .merge(danger_events[key_cols + ["hd_shot", "md_shot", "ld_shot", "is_rebound"]], on=key_cols, how="left")
        .merge(rush_events[key_cols + ["rush_shot"]], on=key_cols, how="left")
    )

    combined["hd_shot"] = combined["hd_shot"].fillna(0).astype(int)
    combined["md_shot"] = combined["md_shot"].fillna(0).astype(int)
    combined["ld_shot"] = combined["ld_shot"].fillna(0).astype(int)
    combined["is_rebound"] = combined["is_rebound"].fillna(0).astype(int)
    combined["rush_shot"] = combined["rush_shot"].fillna(0).astype(int)

    return combined

### Scoring Chances

In [48]:
def add_scoring_chances(df):
    """
    Adds scoring chance classifications at the EVENT level.

    Produces:
      - base_scoring_chance_score (quality only)
      - adj_scoring_chance_score (quality + context)
      - scoring_chance (binary, base definition)
      - rush_scoring_chance (contextual)
    """
    d = df.copy()

    # -------------------------------------------------
    # Order events (important for prev_team_id)
    # -------------------------------------------------
    if "sortOrder" in d.columns:
        d = d.sort_values(["game_id", "sortOrder"])

    # # -------------------------------------------------
    # # Previous team (for AGAINST attribution later)
    # # -------------------------------------------------
    # d["opponent_team_id"] = d.groupby("game_id")["team_id"].shift(1)

    # -------------------------------------------------
    # BASE scoring chance score (NO context)
    # HD=3, MD=2, LD=1
    # -------------------------------------------------
    d["base_scoring_chance_score"] = (
        3 * d["hd_shot"]
        + 2 * d["md_shot"]
        + 1 * d["ld_shot"]
    )

    # -------------------------------------------------
    # BASE binary scoring chance
    # (this is what SCF / SCA should use)
    # -------------------------------------------------
    d["scoring_chance"] = (
        d["base_scoring_chance_score"] >= 2
    ).astype(int)

    # -------------------------------------------------
    # CONTEXTUAL bonuses / penalties
    # -------------------------------------------------
    d["context_bonus"] = (
        (d["is_rebound"] == 1).astype(int)
        + (d["rush_shot"] == 1).astype(int)
    )

    d["blocked_penalty"] = (
        d["type_code"] == 508
    ).astype(int)

    # -------------------------------------------------
    # ADJUSTED scoring chance score
    # (use for weighting, not counting)
    # -------------------------------------------------
    d["adj_scoring_chance_score"] = (
        d["base_scoring_chance_score"]
        + d["context_bonus"]
        - d["blocked_penalty"]
    )

    # -------------------------------------------------
    # Outcome flags
    # -------------------------------------------------
    d["scoring_chance_goal"] = (
        d["type_code"] == 505
    ).astype(int)

    d["shot_on_goal"] = (
        (d["type_code"] == 506).astype(int)
        + d["scoring_chance_goal"]
    )

    # -------------------------------------------------
    # Contextual rush scoring chance (subset)
    # -------------------------------------------------
    d["rush_scoring_chance"] = (
        (d["scoring_chance"] == 1)
        & (d["rush_shot"] == 1)
    ).astype(int)

    return d


#### Combined data

In [ ]:
hist_combined_chances_data = hist_shot_events.merge(hist_danger_event[['season_id', 'game_id', 'team_id', 'sortOrder', 'hd_shot', 'md_shot', 'ld_shot', 'is_rebound']], 
                                               on =  ['season_id', 'game_id', 'team_id', 'sortOrder'], how = "left")

hist_combined_chances_data = hist_combined_chances_data.merge(hist_rush_events[['season_id', 'game_id', 'team_id', 'sortOrder', 'rush_shot']], 
                                               on =  ['season_id', 'game_id', 'team_id', 'sortOrder'], how = "left")



In [49]:

curr_combined_chances_data = curr_shot_events.merge(curr_danger_event[['season_id', 'game_id', 'team_id', 'sortOrder', 'hd_shot', 'md_shot', 'ld_shot', 'is_rebound']], 
                                               on =  ['season_id', 'game_id', 'team_id', 'sortOrder'], how = "left")

curr_combined_chances_data = curr_combined_chances_data.merge(curr_rush_events[['season_id', 'game_id', 'team_id', 'sortOrder', 'rush_shot']], 
                                               on =  ['season_id', 'game_id', 'team_id', 'sortOrder'], how = "left")


In [50]:
# hist_scoring_chances_event = add_scoring_chances(hist_combined_chances_data)

curr_scoring_chances_event = add_scoring_chances(curr_combined_chances_data)


In [51]:
curr_scoring_chances_event.head()

,game_id,team_id,home_team,away_team,type_code,typeDescKey,zone_code,x_coord,y_coord,period_number,game_in_seconds,goalie,sortOrder,season_id,shot_attempt,opponent_team_id,hd_shot,md_shot,ld_shot,is_rebound,rush_shot,base_scoring_chance_score,scoring_chance,context_bonus,blocked_penalty,adj_scoring_chance_score,scoring_chance_goal,shot_on_goal,rush_scoring_chance
0,2025020001,13,13,16,508,blocked-shot,D,-61.0,3.0,1,22,<NA>,12,20252026,1,16,0,1,0,0,NaN,2,1,0,1,1,0,0,0
1,2025020001,13,13,16,508,blocked-shot,D,-64.0,-16.0,1,73,<NA>,25,20252026,1,16,0,1,0,0,1.0,2,1,1,1,2,0,0,1
2,2025020001,13,13,16,506,shot-on-goal,O,-58.0,-22.0,1,76,8481519,29,20252026,1,16,0,1,0,1,NaN,2,1,1,0,3,0,1,0
3,2025020001,13,13,16,506,shot-on-goal,O,-33.0,-19.0,1,95,8481519,35,20252026,1,16,0,0,1,0,NaN,1,0,0,0,1,0,1,0
4,2025020001,16,13,16,507,missed-shot,O,55.0,-29.0,1,104,8475683,38,20252026,1,13,0,0,1,0,NaN,1,0,0,0,1,0,0,0


### Danger Corsi and SCF Stats

In [52]:
def build_danger_corsi_and_scf_stats(df):
    """
    Produces:
        - HDCF, HDCA
        - MDCF, MDCA
        - LDCF, LDCA
        - SCF (scoring chances for)
        - SCSF (scoring-chance shots for)
        - SCGF (scoring-chance goals for)
        - SCA (scoring chances against)

    Requires df already to contain:
        hd_shot, md_shot, ld_shot
        rush_shot, is_rebound
        scoring_chance
        opponent_team_id
    """

    d = df.copy()

    # -----------------------------
    # Assign shooting team correctly
    # -----------------------------
    d["shooting_team_id"] = d["team_id"]

    mask_block = d["type_code"] == 508
    d.loc[mask_block, "shooting_team_id"] = d.loc[mask_block, "opponent_team_id"]

    # -----------------------------
    # Scoring chance shot + goal flags
    # -----------------------------
    d["shot_on_goal"] = d["type_code"].isin([505, 506]).astype(int)
    d["scoring_chance_goal"] = (d["type_code"] == 505).astype(int)

    # -----------------------------
    # FOR — scoring chances
    # -----------------------------
    scf = (
        d.groupby(["season_id", "game_id", "shooting_team_id"], as_index=False)
         .agg(
             SCF=("scoring_chance", "sum"),
             SCSF=("shot_on_goal", "sum"),
             SCGF=("scoring_chance_goal", "sum"),
         )
         .rename(columns={"shooting_team_id": "team_id"})
    )

    # -----------------------------
    # AGAINST — scoring chances (opponent-based)
    # -----------------------------
    sca = (
        d.groupby(["season_id", "game_id", "opponent_team_id"], as_index=False)
         .agg(SCA=("scoring_chance", "sum"))
         .rename(columns={"opponent_team_id": "team_id"})
    )

    # -----------------------------
    # Danger-split Corsi FOR
    # -----------------------------
    d["hd_corsi"] = d["hd_shot"]
    d["md_corsi"] = d["md_shot"]
    d["ld_corsi"] = d["ld_shot"]

    corsi_for = (
        d.groupby(["season_id", "game_id", "team_id"], as_index=False)
         .agg(
             HDCF=("hd_corsi", "sum"),
             MDCF=("md_corsi", "sum"),
             LDCF=("ld_corsi", "sum"),
         )
    )

    # -----------------------------
    # Danger-split Corsi AGAINST (opponent-based)
    # -----------------------------
    corsi_against = (
        d.groupby(["season_id", "game_id", "opponent_team_id"], as_index=False)
         .agg(
             HDCA=("hd_corsi", "sum"),
             MDCA=("md_corsi", "sum"),
             LDCA=("ld_corsi", "sum"),
         )
         .rename(columns={"opponent_team_id": "team_id"})
    )

    # -----------------------------
    # FINAL MERGE
    # -----------------------------
    final = (
        scf
        .merge(sca, on=["season_id", "game_id", "team_id"], how="left")
        .merge(corsi_for, on=["season_id", "game_id", "team_id"], how="left")
        .merge(corsi_against, on=["season_id", "game_id", "team_id"], how="left")
        .fillna(0)
    )

    # -----------------------------
    # Percentages
    # -----------------------------
    final["HDCF_pct"] = final["HDCF"] / (final["HDCF"] + final["HDCA"]).replace(0, np.nan)
    final["HDCA_pct"] = final["HDCA"] / (final["HDCF"] + final["HDCA"]).replace(0, np.nan)

    final["MDCF_pct"] = final["MDCF"] / (final["MDCF"] + final["MDCA"]).replace(0, np.nan)
    final["MDCA_pct"] = final["MDCA"] / (final["MDCF"] + final["MDCA"]).replace(0, np.nan)

    final["LDCF_pct"] = final["LDCF"] / (final["LDCF"] + final["LDCA"]).replace(0, np.nan)
    final["LDCA_pct"] = final["LDCA"] / (final["LDCF"] + final["LDCA"]).replace(0, np.nan)

    return final

In [53]:
curr_danger_scoring_stats = build_danger_corsi_and_scf_stats(curr_scoring_chances_event)


# hist_danger_scoring_stats = build_danger_corsi_and_scf_stats(hist_scoring_chances_event)

In [54]:
curr_danger_scoring_stats.head()

,season_id,game_id,team_id,SCF,SCSF,SCGF,SCA,HDCF,MDCF,LDCF,HDCA,MDCA,LDCA,HDCF_pct,HDCA_pct,MDCF_pct,MDCA_pct,LDCF_pct,LDCA_pct
0,20252026,2025020001,13,37,37,3,24,17,24,28,11,14,26,0.607143,0.392857,0.631579,0.368421,0.518519,0.481481
1,20252026,2025020001,16,28,19,2,41,11,14,26,17,24,28,0.392857,0.607143,0.368421,0.631579,0.481481,0.518519
2,20252026,2025020002,3,34,25,0,35,14,19,29,18,17,26,0.437500,0.562500,0.527778,0.472222,0.527273,0.472727
3,20252026,2025020002,5,32,31,3,31,18,17,26,14,19,29,0.562500,0.437500,0.472222,0.527778,0.472727,0.527273
4,20252026,2025020003,21,27,23,4,29,13,12,26,15,17,37,0.464286,0.535714,0.413793,0.586207,0.412698,0.587302


In [ ]:
hist_shots_stats.head()

#### Cumulative Scoring Chances

In [55]:
def cumulative_scoring_chances(danger_scoring: pd.DataFrame,
                               corsi_stats: pd.DataFrame) -> pd.DataFrame:
    scoring = danger_scoring.copy()
    shots = corsi_stats.copy()

    # -------------------------------------------------
    # Sort both dataframes consistently
    # -------------------------------------------------
    sort_cols = (
        ["season_id", "team_id", "game_date", "game_id"]
        if "game_date" in scoring.columns
        else ["season_id", "team_id", "game_id"]
    )

    scoring = scoring.sort_values(sort_cols)
    shots = shots.sort_values(sort_cols)

    # -------------------------------------------------
    # Grouping
    # -------------------------------------------------
    g_sc = scoring.groupby(["season_id", "team_id"])
    g_cf = shots.groupby(["season_id", "team_id"])

    # -------------------------------------------------
    # Cumulative scoring-chance metrics
    # -------------------------------------------------
    scoring["c_SCF"] = g_sc["SCF"].cumsum()
    scoring["c_SCSF"] = g_sc['SCSF'].cumsum()
    scoring['c_SCGF'] = g_sc['SCGF'].cumsum()
    scoring['c_SCA'] = g_sc['SCA'].cumsum()
    

    scoring["SCF_pg"] = scoring["c_SCF"] / scoring["games_played"]
    scoring["SCA_pg"] = scoring["c_SCA"] / scoring["games_played"]
    
    
    scoring["c_HDCF"] = g_sc["HDCF"].cumsum()
    scoring["c_HDCA"] = g_sc["HDCA"].cumsum()
    scoring["c_MDCF"] = g_sc["MDCF"].cumsum()
    scoring["c_MDCA"] = g_sc["MDCA"].cumsum()
    scoring["c_LDCF"] = g_sc["LDCF"].cumsum()
    scoring["c_LDCA"] = g_sc["LDCA"].cumsum()


    # -------------------------------------------------
    # Cumulative Corsi metrics
    # -------------------------------------------------
    shots["c_corsi_for"] = g_cf["corsi_for"].cumsum()
    shots["c_corsi_against"] = g_cf["corsi_against"].cumsum()
    shots["c_corsi_diff"] = (
        shots["c_corsi_for"] - shots["c_corsi_against"]
    )


    shots["corsi_diff_per_game"] = (
        shots["c_corsi_diff"] / shots["games_played"]
    )




    # -------------------------------------------------
    # Merge back together
    # -------------------------------------------------
    merge_cols = ["season_id", "team_id", "game_id"]
    if "game_date" in scoring.columns and "game_date" in shots.columns:
        merge_cols.append("game_date")

    out = scoring.merge(
        shots[
            merge_cols + [
                "c_corsi_for",
                "c_corsi_against",
                "c_corsi_diff",
                "corsi_diff_per_game",
                

            ]
        ],
        on=merge_cols,
        how="left",
    )
    out = out.drop(columns=['games_played'])

    return out


In [56]:
# hist_cumulative_scoring_chances =cumulative_scoring_chances(hist_danger_scoring_stats, hist_shots_stats)
curr_cumulative_scoring_chances =cumulative_scoring_chances(curr_danger_scoring_stats,curr_shots_stats)

curr_cumulative_scoring_chances.head()

KeyError: 'games_played'

### Shooting Percentage (shots on goal)

In [ ]:
def build_team_shooting_stats(df):
    df = df.copy()

    df["goal"]  = df['scoring_chance_goal']
    df["sog"]   = df['shot_on_goal']


    df["hd_sog"] = ((df["sog"] == 1) & (df["hd_shot"] == 1)).astype(int)
    df["md_sog"] = ((df["sog"] == 1) & (df["md_shot"] == 1)).astype(int)
    df["ld_sog"] = ((df["sog"] == 1) & (df["ld_shot"] == 1)).astype(int)

    df["hd_goal"] = ((df["goal"] == 1) & (df["hd_shot"] == 1)).astype(int)
    df["md_goal"] = ((df["goal"] == 1) & (df["md_shot"] == 1)).astype(int)
    df["ld_goal"] = ((df["goal"] == 1) & (df["ld_shot"] == 1)).astype(int)

    opp_map = (
        df[["game_id","team_id"]].drop_duplicates()
          .groupby("game_id")["team_id"].apply(list)
    )
    opponent = {
        gid: ({t[0]:t[1], t[1]:t[0]} if len(t)==2 else {})
        for gid, t in opp_map.items()
    }
    df["opponent_team_id"] = df.apply(
        lambda r: opponent.get(r["game_id"],{}).get(r["team_id"]),
        axis=1
    )

    stats_for = (
        df.groupby(["season_id", "game_id","team_id"], as_index=False)
          .agg(
              shots_for=("sog","sum"),
              goals_for=("goal","sum"),
              hd_sog=("hd_sog","sum"),
              md_sog=("md_sog","sum"),
              ld_sog=("ld_sog","sum"),
              hd_goals=("hd_goal","sum"),
              md_goals=("md_goal","sum"),
              ld_goals=("ld_goal","sum"),
          )
    )
    stats_for["shot_pct"] = 100 * stats_for["goals_for"] / stats_for["shots_for"].replace(0,np.nan)

    stats_against = (
        df.groupby(["season_id", "game_id","opponent_team_id"], as_index=False)
          .agg(
              shots_against=("sog","sum"),
              goals_against=("goal","sum"),
              hd_against=("hd_sog","sum"),
              md_against=("md_sog","sum"),
              ld_against=("ld_sog","sum"),
          )
          .rename(columns={"opponent_team_id":"team_id"})
    )

    stats = stats_for.merge(stats_against, on=["season_id", "game_id","team_id"], how="left")

    stats["sv_pct"] = 100 - (100 * stats["goals_against"] / stats["shots_against"].replace(0,np.nan))
    stats["pdo"] = stats["shot_pct"] + stats["sv_pct"]
    
    
    stats = stats.sort_values(
    ["season_id", "team_id", "game_id"]
)

    stats["games_played"] = (
        stats
        .groupby(["season_id", "team_id"])
        .cumcount() + 1
    )

    return stats


In [ ]:
curr_sog_stats = build_team_shooting_stats(curr_scoring_chances_event)
hist_sog_stats = build_team_shooting_stats(hist_scoring_chances_event)

In [ ]:
curr_sog_stats.head()

### Cumulative SOG 

In [ ]:
def cumulative_sog(df: pd.DataFrame) -> pd.DataFrame:

    sog = df.copy()

    # -------------------------------------------------
    # Sort BEFORE cumulative math
    # -------------------------------------------------
    sort_cols = (
        ["season_id", "team_id", "game_date", "game_id"]
        if "game_date" in sog.columns
        else ["season_id", "team_id", "game_id"]
    )
    sog = sog.sort_values(sort_cols)

    # -------------------------------------------------
    # Group by season + team
    # -------------------------------------------------
    g = sog.groupby(["season_id", "team_id"])


    # -------------------------------------------------
    # Cumulative FOR
    # -------------------------------------------------
    sog["c_shots_for"]  = g["shots_for"].cumsum()
    sog["c_goals_for"]  = g["goals_for"].cumsum()

    sog["c_hd_sog"] = g["hd_sog"].cumsum()
    sog["c_md_sog"] = g["md_sog"].cumsum()
    sog["c_ld_sog"] = g["ld_sog"].cumsum()

    sog["c_hd_goals"] = g["hd_goals"].cumsum()
    sog["c_md_goals"] = g["md_goals"].cumsum()
    sog["c_ld_goals"] = g["ld_goals"].cumsum()

    # -------------------------------------------------
    # Cumulative AGAINST
    # -------------------------------------------------
    sog["c_shots_against"]  = g["shots_against"].cumsum()
    sog["c_goals_against"]  = g["goals_against"].cumsum()

    sog["c_hd_against"] = g["hd_against"].cumsum()
    sog["c_md_against"] = g["md_against"].cumsum()
    sog["c_ld_against"] = g["ld_against"].cumsum()

    # -------------------------------------------------
    # Recompute percentages (DO NOT cumsum these)
    # -------------------------------------------------
    sog["c_shot_pct"] = np.where(
        sog["c_shots_for"] > 0,
        sog["c_goals_for"] / sog["c_shots_for"],
        0.0
    )

    sog["c_sv_pct"] = np.where(
        sog["c_shots_against"] > 0,
        1 - (sog["c_goals_against"] / sog["c_shots_against"]),
        0.0
    )

    sog["c_pdo"] = (sog["c_shot_pct"] + sog["c_sv_pct"]) * 100
    sog=sog.drop(columns=['games_played'])

    return sog

    


In [ ]:

hist_cumulative_sog = cumulative_sog(hist_sog_stats)
curr_cumulative_sog = cumulative_sog(curr_sog_stats)

curr_cumulative_sog.head()

## Strength of Schedule - need to update this re: off/fut games

In [ ]:
def strength_of_schedule(df):
    sos = df.copy()

    games_sos = []

    # -------------------------------------------------
    # Build two rows per game (HOME + AWAY)
    # -------------------------------------------------
    for _, row in sos.iterrows():

        # AWAY team row
        games_sos.append({
            "season_id": row["season_id"],
            "game_id": int(row["game_id"]),
            "team_id": int(row["away_team"]),
            "tri_code": row["away_tri_code"],
            "opponent_id": int(row["home_team"]),
            "opponent": row["home_tri_code"],
            "goals_for": row["awayTeam.score"],
            "goals_against": row["homeTeam.score"],
        })

        # HOME team row
        games_sos.append({
            "season_id": row["season_id"],
            "game_id": int(row["game_id"]),
            "team_id": int(row["home_team"]),
            "tri_code": row["home_tri_code"],
            "opponent_id": int(row["away_team"]),
            "opponent": row["away_tri_code"],
            "goals_for": row["homeTeam.score"],
            "goals_against": row["awayTeam.score"],
        })

    games_sos = pd.DataFrame(games_sos)

    # -------------------------------------------------
    # Sort + win calculation
    # -------------------------------------------------
    games_sos = games_sos.sort_values(
        ["season_id", "team_id", "tri_code", "game_id"]
    )

    games_sos["win"] = (
        games_sos["goals_for"] > games_sos["goals_against"]
    ).astype(int)

    # -------------------------------------------------
    # Wins BEFORE (leave NaN for first game)
    # -------------------------------------------------
    games_sos["wins_before"] = (
        games_sos
        .groupby(["season_id", "team_id", "tri_code"])["win"]
        .transform(lambda s: s.cumsum().shift(1))
        .astype("Int64")
    )


    # -------------------------------------------------
    # Games BEFORE / played
    # -------------------------------------------------
    games_sos["games_before"] = (
        games_sos
        .groupby(["season_id", "team_id", "tri_code"])
        .cumcount()
    )

    games_sos["games_played"] = games_sos["games_before"] + 1

    # -------------------------------------------------
    # Win% BEFORE (NaN if no prior games)
    # -------------------------------------------------
    games_sos["win_pct_before"] = np.where(
        games_sos["games_before"] > 0,
        games_sos["wins_before"] / games_sos["games_before"],
        np.nan
    )

    # -------------------------------------------------
    # Deduplicate opponent win% BEFORE merge
    # -------------------------------------------------
    opp_win = (
        games_sos[["season_id", "tri_code", "game_id", "win_pct_before"]]
        .drop_duplicates(subset=["season_id", "tri_code", "game_id"])
    )

    games_sos = games_sos.merge(
        opp_win.rename(columns={
            "tri_code": "opponent",
            "win_pct_before": "opponent_win_pct_before"
        }),
        on=["season_id", "opponent", "game_id"],
        how="left"
    )

    # -------------------------------------------------
    # SOS — PAST (NaN until at least one prior game)
    # -------------------------------------------------
    games_sos["sos_past"] = (
        games_sos
        .groupby(["season_id", "tri_code"])["opponent_win_pct_before"]
        .expanding()
        .mean()
        .reset_index(level=[0, 1], drop=True)
    )

    # -------------------------------------------------
    # SOS — FUTURE (reverse expanding, valid from game 1)
    # -------------------------------------------------
    games_sos["sos_future"] = (
        games_sos.iloc[::-1]
        .groupby(["season_id", "team_id", "tri_code"])["opponent_win_pct_before"]
        .expanding()
        .mean()
        .reset_index(level=[0, 1, 2], drop=True)
        .iloc[::-1]
    )

    # -------------------------------------------------
    # Final cleanup
    # -------------------------------------------------
    games_sos = (
        games_sos
        .drop(columns=['opponent_id', 'opponent', 'goals_for', 'goals_against', 'opponent_win_pct_before'])
        .sort_values(by=["season_id", "game_id"])
    )

    return games_sos


In [ ]:
hist_games_sos = strength_of_schedule(history_game_summary)
curr_games_sos = strength_of_schedule(game_summary)


## Goalies

In [ ]:
history_goalies = con.execute("""
                              SELECT *
                              FROM all_goalie_games
                              WHERE season_id !=20252026
                              ORDER BY goalie_name""").df()

history_goalies.head()

In [ ]:
goalies = con.execute("""
                      SELECT *
                      FROM all_goalie_games
                      WHERE season_id =20252026
                      ORDER BY goalie_name""").df()

goalies.head()

In [ ]:
def build_team_goalie_stats(goalies_df, teams_df):

    # -----------------------------------------------------
    # Helpers
    # -----------------------------------------------------
    def safe_weighted_mean(values, weights):
        weights = weights.replace(0, np.nan)
        num = (values * weights).sum()
        den = weights.sum()
        return num / den if den > 0 else np.nan

    g = goalies_df.copy()

    # -----------------------------------------------------
    # 1. Individual goalie performance per game
    # -----------------------------------------------------
    goalies_indiv = (
        g.groupby(["goalie_name","season_id", "tri_code", "game_id"])
         .agg(
             i_savePct=("save_pct",
                        lambda x: safe_weighted_mean(
                            x, g.loc[x.index, "shots_against"]
                        )),
             i_GAA=("gaa",
                    lambda x: safe_weighted_mean(
                        x, g.loc[x.index, "toi"]
                    )),
             i_gamesPlayed=("games_played", "sum"),
             i_gamesStarted=("games_started", "sum"),
             i_wins=("wins", "sum"),
             i_losses=("losses", "sum"),
             i_otlosses=("ot_losses", "sum"),
             i_shutouts=("shutouts", "sum"),
             i_saves=("saves", "sum"),
             i_shotsAgainst=("shots_against", "sum"),
             i_timeOnIce=("toi", "sum"),
         )
         .reset_index()
    )

    # -----------------------------------------------------
    # 2. Aggregate to team-level for each game
    # -----------------------------------------------------
    game_goalie_stats = (
        goalies_indiv.groupby(["season_id","game_id", "tri_code"])
        .agg(
            game_savePct=("i_savePct",
                          lambda x: safe_weighted_mean(
                              x, goalies_indiv.loc[x.index, "i_shotsAgainst"]
                          )),
            game_GAA=("i_GAA",
                      lambda x: safe_weighted_mean(
                          x, goalies_indiv.loc[x.index, "i_timeOnIce"]
                      )),
            game_wins=("i_wins", "sum"),
            game_losses=("i_losses", "sum"),
            game_shutouts=("i_shutouts", "sum"),
            game_saves=("i_saves", "sum"),
            game_shotsAgainst=("i_shotsAgainst", "sum"),
        )
        .reset_index()
    )

    # -----------------------------------------------------
    # 3. Output formatting
    # -----------------------------------------------------
    output = game_goalie_stats[["season_id", "game_id", "tri_code", "game_savePct", "game_GAA" ]]
    output = output.merge(teams_df, on="tri_code", how="left")

    return output


In [ ]:
curr_cumulative_goalie_stats = build_team_goalie_stats(goalies, teams)
hist_cumulative_goalie_stats = build_team_goalie_stats(history_goalies, teams)


In [ ]:
curr_cumulative_goalie_stats.head()

## Special Teams

In [ ]:
history_pk = con.execute("""
                         SELECT all_pk.*, all_teams.tri_code
                         FROM all_pk
                         join all_teams 
                         on all_teams.team_id = all_pk.team_id
                         WHERE season_id != 20252026
                         ORDER BY game_id""").df()
history_pk.head()

In [ ]:
current_pk = con.execute(""" 
                         SELECT  all_pk.*, all_teams.tri_code
                         FROM all_pk
                         join all_teams 
                         on all_teams.team_id = all_pk.team_id
                         WHERE season_id = 20252026
                         ORDER BY game_id""").df()
current_pk.head()

In [ ]:
history_pp = con.execute(""" 
                         SELECT  all_pp.*, all_teams.tri_code
                         FROM all_pp
                        join all_teams 
                         on all_teams.team_id = all_pp.team_id
                         WHERE season_id != 20252026
                         ORDER BY game_id""").df()
history_pp.head()

In [ ]:
current_pp = con.execute(""" 
                         SELECT all_pp.*, all_teams.tri_code
                         FROM all_pp
                         join all_teams 
                         on all_teams.team_id = all_pp.team_id
                         WHERE season_id = 20252026
                         ORDER BY game_id""").df()
current_pp.head()

In [ ]:
def build_special_teams_stats(pp_df, pk_df):
    import pandas as pd
    import numpy as np

    pp = pp_df.copy()
    pk = pk_df.copy()

    # ------------------------------------------------------------------
    # 1. Clean + order
    # ------------------------------------------------------------------
    pp = pp.sort_values(["season_id", "team_id", "game_date"])
    pk = pk.sort_values(["season_id", "team_id", "game_date"])

    
    pp = pp.sort_values(
        ["season_id", "team_id", "game_id"]
    )

    pp["games_played"] = (
        pp
        .groupby(["season_id", "team_id"])
        .cumcount() + 1
    )

    # ------------------------------------------------------------------
    # 2. Merge PP + PK at game level
    # ------------------------------------------------------------------
    merged = pp.merge(
        pk,
        on=["season_id", "game_id", "team_id", "game_date"],
        how="inner",
        suffixes=("_pp", "_pk")
    )

    # ==================================================================
    # ========================= POWER PLAY ==============================
    # ==================================================================
    pp_rank = pp[
        [
            "season_id",
            "game_date",
            "team_id",
            "tri_code",
            "games_played",
            "pp_net_g",
            "pp_opportunities",
        ]
    ].copy()

    pp_rank["c_pp_net_g"] = (
        pp_rank.groupby(["season_id", "team_id"])["pp_net_g"].cumsum()
    )

    pp_rank["c_pp_opps"] = (
        pp_rank.groupby(["season_id", "team_id"])["pp_opportunities"].cumsum()
    )

    pp_rank["net_pp_pct"] = np.where(
        pp_rank["c_pp_opps"] > 0,
        pp_rank["c_pp_net_g"] / pp_rank["c_pp_opps"],
        0
    )

    # ==================================================================
    # ======================== PENALTY KILL =============================
    # ==================================================================
    pk_rank = pk[
        [
            "season_id",
            "game_date",
            "team_id",
            "pp_ga",
            "times_sh",
        ]
    ].copy()

    pk_rank["c_pp_ga"] = (
        pk_rank.groupby(["season_id", "team_id"])["pp_ga"].cumsum()
    )

    pk_rank["c_times_sh"] = (
        pk_rank.groupby(["season_id", "team_id"])["times_sh"].cumsum()
    )

    pk_rank["net_pk_pct"] = np.where(
        pk_rank["c_times_sh"] > 0,
        pk_rank["c_pp_ga"] / pk_rank["c_times_sh"],
        0
    )

    # ------------------------------------------------------------------
    # 3. Build full season-date-team grid
    # ------------------------------------------------------------------
    all_dates = (
        pp_rank.groupby("season_id")["game_date"]
        .unique()
        .explode()
        .unique()
    )

    teams = pp_rank["team_id"].unique()

    grid = pd.MultiIndex.from_product(
        [pp_rank["season_id"].unique(), all_dates, teams],
        names=["season_id", "game_date", "team_id"]
    )

    # ------------------------------------------------------------------
    # 4. Reindex + forward fill (cumulative only)
    # ------------------------------------------------------------------
    pp_full = (
        pp_rank
        .set_index(["season_id", "game_date", "team_id"])
        .reindex(grid)
        .sort_index()
    )

    pk_full = (
        pk_rank
        .set_index(["season_id", "game_date", "team_id"])
        .reindex(grid)
        .sort_index()
    )

    for col in ["c_pp_net_g", "c_pp_opps", "net_pp_pct"]:
        pp_full[col] = pp_full.groupby(["season_id", "team_id"])[col].ffill()

    for col in ["c_pp_ga", "c_times_sh", "net_pk_pct"]:
        pk_full[col] = pk_full.groupby(["season_id", "team_id"])[col].ffill()

    # ------------------------------------------------------------------
    # 5. Percent cleanup + daily ranks
    # ------------------------------------------------------------------
    pp_full["net_pp_pct"] = (pp_full["net_pp_pct"] * 100).round(1).fillna(0)
    pk_full["net_pk_pct"] = (pk_full["net_pk_pct"] * 100).round(1).fillna(0)

    pp_full["ppng_rank"] = (
        pp_full.groupby(["season_id", "game_date"])["net_pp_pct"]
        .rank(method="dense", ascending=False)
        .astype(int)
    )

    pk_full["pk_rank"] = (
        pk_full.groupby(["season_id", "game_date"])["net_pk_pct"]
        .rank(method="dense", ascending=True)
        .astype(int)
    )

    # ------------------------------------------------------------------
    # 6. Merge cumulative stats + ranks back to game rows
    # ------------------------------------------------------------------
    pp_features = pp_full.reset_index()[
        ["season_id", "game_date", "team_id",
         "c_pp_opps", "c_pp_net_g", "net_pp_pct", "ppng_rank"]
    ]

    pk_features = pk_full.reset_index()[
        ["season_id", "game_date", "team_id",
         "c_pp_ga", "c_times_sh", "net_pk_pct", "pk_rank"]
    ]

    special_teams = (
        merged
        .merge(pp_features, on=["season_id", "game_date", "team_id"], how="left")
        .merge(pk_features, on=["season_id", "game_date", "team_id"], how="left")
    )

    # ------------------------------------------------------------------
    # 7. Drop unused columns
    # ------------------------------------------------------------------
    special_teams = special_teams.drop(
        columns=[
            'home_road_pp', 'losses_pp', 'opponent_pp', 'ot_losses_pp',
            'point_pct', 'points_pp', 'team_name_pp', 'ties_pp', 'wins_pp',
            'games_played_pk', 'home_road_pk', 'losses_pk', 'opponent_pk',
            'ot_losses_pk', 'points_pk', 'points_pct', 'team_name_pk',
            'ties_pk', 'wins_pk', 'tri_code_pk', 'games_played_pp'
        ],
        errors="ignore"
    )

    # Rename for consistency
    special_teams = special_teams.rename(
        columns={"pp_opportunities": "pp_opp", "tri_code_pp":"tri_code"}
    )

    # ------------------------------------------------------------------
    # 8. Column ordering
    # ------------------------------------------------------------------
    col_order = [
        'season_id', 'game_id', 'team_id', 'tri_code',

        # PP stats
        'pp_opp', 'pp_opp_pg', 'pp_toi',
        'pp_gf', 'pp_gpg',
        'pp_net_g', 'pp_net_gpg',
        'pp_pct', 'pp_net_pct',
        'sh_ga', 'sh_ga_pg',

        # PK stats
        'times_sh', 'times_sh_pg',
        'pk_net_pct', 'pk_pct',
        'pk_net_g', 'pk_net_gpg', 'pk_toi_pg',
        'pp_ga', 'pp_ga_pg',
        'sh_gf', 'sh_gf_pg',

        # cumulative stats
        'c_pp_opps', 'c_pp_net_g', 'net_pp_pct', 'ppng_rank',
        'c_pp_ga', 'c_times_sh', 'net_pk_pct', 'pk_rank',
    ]

    ordered_cols = [c for c in col_order if c in special_teams.columns]
    remaining_cols = [c for c in special_teams.columns if c not in ordered_cols]

    special_teams = special_teams[ordered_cols + remaining_cols]

    return special_teams


In [ ]:
current_pk.columns

In [ ]:
hist_special_teams = build_special_teams_stats(history_pp, history_pk)
curr_special_teams = build_special_teams_stats(current_pp, current_pk)

curr_special_teams.head()

## Final DFs

In [ ]:
history_games = history_pbp[['season_id','game_id','team_id']].drop_duplicates()
history_games=history_games[history_games['team_id'].notna()].reset_index(drop=True)
history_games.head()

In [ ]:
hist_cumulative_scoring_chances.columns

In [ ]:
def coalesce_duplicates(df, left="_x", right="_y"):
    for col in df.columns:
        if col.endswith(left):
            base = col[:-len(left)]
            other = base + right
            if other in df.columns:
                df[base] = df[col].combine_first(df[other])
                df.drop([col, other], axis=1, inplace=True)
    return df


history_df = (
    history_games
    .merge(history_game_summary[['game_id', 'game_date']], on='game_id', how='inner')
    .merge(hist_games_sos, on=['season_id', 'game_id', 'team_id'], how='inner')
    .merge(hist_special_teams, on=['season_id', 'game_id', 'team_id','tri_code'], how='inner')
    .merge(
        hist_cumulative_goalie_stats[['game_id', 'team_id', 'game_savePct', 'game_GAA']],
        on=['game_id', 'team_id'],
        how='inner'
    )
    .merge(history_cumulative_rush, on=['season_id','game_id', 'team_id'], how='inner')
    .merge(hist_cumulative_shots, on=['season_id','game_id', 'team_id'], how = 'inner' )
    .merge(hist_cumulative_scoring_chances, on=['season_id', 'game_id', 'team_id'], how='left')
    .merge(hist_cumulative_sog, on=['season_id', 'game_id', 'team_id',], how='inner' )

)
history_df = coalesce_duplicates(history_df)
history_df.head()

In [ ]:
def coalesce_dupes(df, left="_x", right="_y"):
    for col in df.columns:
        if col.endswith(left):
            base = col[:-len(left)]
            other = base + right
            if other in df.columns:
                df[base] = df[col].combine_first(df[other])
                df.drop([col, other], axis=1, inplace=True)
    return df
history_df = history_df.merge(history_daily_standings, on=[ 'season_id', 'team_id','tri_code','game_date'], how = 'inner')
history_df = coalesce_dupes(history_df)
history_df.head()

In [ ]:
cols = list(history_df.columns)

# remove team_id from its current position
cols.remove("game_date")
cols.remove("team_name")


# insert it at position 2 (3rd column, excluding index)
cols.insert(2, "game_date")
cols.insert(5, "team_name")

# reassign columns
history_df = history_df[cols]
history_df.head(2)

In [ ]:
current_games = current_pbp[['season_id','game_id','team_id']].drop_duplicates()
current_games=current_games[current_games['team_id'].notna()].reset_index(drop=True)
current_games.head()

In [ ]:
def coalesce_duplicates(df, left="_x", right="_y"):
    for col in df.columns:
        if col.endswith(left):
            base = col[:-len(left)]
            other = base + right
            if other in df.columns:
                df[base] = df[col].combine_first(df[other])
                df.drop([col, other], axis=1, inplace=True)
    return df

curr_df = (
    current_games
    .merge(game_summary[['game_id', 'game_date']], on='game_id', how='inner')
    .merge(curr_games_sos, on=['season_id', 'game_id', 'team_id'], how='inner')
    .merge(curr_special_teams, on=['season_id', 'game_id', 'team_id','tri_code'], how='inner')
    .merge(
        curr_cumulative_goalie_stats[['game_id', 'team_id', 'game_savePct', 'game_GAA']],
        on=['game_id', 'team_id'],
        how='inner'
    )
    .merge(current_cumulative_rush, on=['season_id','game_id', 'team_id'], how='inner')
    .merge(curr_cumulative_shots, on=['season_id','game_id', 'team_id'], how = 'inner' )
    .merge(curr_cumulative_scoring_chances, on=['season_id', 'game_id', 'team_id'], how='left')
    .merge(curr_cumulative_sog, on=['season_id', 'game_id', 'team_id',], how='inner' )
)

curr_df = coalesce_duplicates(curr_df)
curr_df.head()

In [ ]:
def coalesce_dupes(df, left="_x", right="_y"):
    for col in df.columns:
        if col.endswith(left):
            base = col[:-len(left)]
            other = base + right
            if other in df.columns:
                df[base] = df[col].combine_first(df[other])
                df.drop([col, other], axis=1, inplace=True)
    return df

curr_df = curr_df.merge(daily_standings, on=[ 'season_id', 'team_id','tri_code','game_date'], how = 'inner')

curr_df = coalesce_dupes(curr_df)
curr_df.head()

In [ ]:
cols = list(curr_df.columns)

# remove team_id from its current position
cols.remove("game_date")
cols.remove("team_name")


# insert it at position 2 (3rd column, excluding index)
cols.insert(2, "game_date")
cols.insert(5, "team_name")


# reassign columns
curr_df = curr_df[cols]
curr_df.head(2)

In [ ]:
#special features

def special_features(df):
    df['ot_wins'] = (df['rotw']-df['rw']).clip(lower=0)
    df['overtimes'] = df['ot_losses']+df['ot_wins']
    df["ot_games_pct"] = (
        df["overtimes"] / df["games_played"]
    ).fillna(0).round(2)


    # Point% × Strength of Schedule
    df["pt_pct_x_sos"] = df["points_pct"] * df["sos_future"]

    # Goalie GA × SV% interaction
    df["gaa_x_sv"] = df["game_GAA"] * df["game_savePct"]

    # Goal differential × Rush differential
    df["gd_x_rushdiff"] = (
        df["goal_diff"] * df["rush_shot_differential"]
    )

    # PK workload × PK%
    df["pk_load_x_pk_pct"] = (
        df["times_sh_pg"] * df["pk_pct"]
    )
    return df
        
history_data = special_features(history_df)
current_data = special_features(curr_df)    
current_data.head()   

In [ ]:
# con.execute("DROP TABLE IF EXISTS history_data")

# con.execute("DROP TABLE IF EXISTS current_data")

# con.execute("""
# CREATE TABLE IF NOT EXISTS history_data AS
# SELECT * FROM history_df
# """)

# con.execute("""
# CREATE TABLE IF NOT EXISTS current_data AS
# SELECT * FROM curr_df
# """)


In [ ]:

history_df = con.execute('select* from history_data').fetch_df()
history_df.head(3)
